<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/x-beam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

#find x-beam nodes' coordinate

In [ ]:
import pandas as pd
from google.colab import files


df_all = pd.read_csv('nodeExport.txt', sep='\t')
df_ids = pd.read_csv('x-beamnew.txt', sep='\t')


df_all = df_all[['Node Number', 'X Location (m)', 'Y Location (m)', 'Z Location (m)']]
df_all.columns = ['Node', 'X', 'Y', 'Z']


df_ids = df_ids[['Node Number']]
df_ids.columns = ['Node']


result = pd.merge(df_ids, df_all, on='Node', how='left')


output_filename = 'matched_nodes_coordinates.txt'
result.to_csv(output_filename, sep='\t', index=False)
print(f"文件已生成: {output_filename}")


files.download(output_filename)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np


df = pd.read_csv('x-beams withcoordinates new.txt', sep='\t')

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')


x_range = df['X'].max() - df['X'].min()
y_range = df['Y'].max() - df['Y'].min()
z_range = df['Z'].max() - df['Z'].min()


ax.scatter(df['X'], df['Y'], df['Z'], c='hotpink', marker='.', s=5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Point Cloud - True Scale (Pink)')


ax.set_box_aspect((x_range, y_range, z_range))


ax.view_init(elev=90, azim=-90)

plt.savefig('3d_point_cloud_pink_true_scale.png')
plt.show()

#classify nodes to every body

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.cm as cm


coords_file = 'x-beams withcoordinates.txt'
map_file   = 'node_solid_map new.txt'

print("1. 正在读取文件...")

try:

    df_coords = pd.read_csv(coords_file, sep='\t')
    df_coords.columns = [c.strip() for c in df_coords.columns]
    print(f"   坐标文件读取成功: {len(df_coords)} 行")


    df_map = pd.read_csv(map_file, sep=',', skipinitialspace=True)
    df_map.columns = [c.strip() for c in df_map.columns]
    print(f"   映射文件读取成功: {len(df_map)} 行")


    df_coords['Node'] = df_coords['Node'].astype(int)
    df_map['NodeID']  = df_map['NodeID'].astype(int)


    df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')


    body_col = 'BodyID'
    unique_bodies = df_merged[body_col].unique()

    unique_bodies = [b for b in unique_bodies if b != 0]

    print(f"✅ 合并成功！")
    print(f"   共筛选出节点: {len(df_merged)}")
    print(f"   识别出独立部件: {len(unique_bodies)} 个")


    print("2. 正在生成 3D 彩色图像...")
    fig = plt.figure(figsize=(15, 12))
    ax = fig.add_subplot(111, projection='3d')


    colors = cm.jet(np.linspace(0, 1, len(unique_bodies)))


    grouped = df_merged.groupby(body_col)

    for i, (body_id, group) in enumerate(grouped):
        if body_id == 0: continue


        ax.scatter(group['X'], group['Y'], group['Z'],
                   s=10,
                   color=colors[i],
                   alpha=1.0,
                   label=f'Body {int(body_id)}' if i < 10 else "")


    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'X-Beams Colored by BodyID ({len(unique_bodies)} Parts)')


    try:
        max_range = max(df_merged['X'].max()-df_merged['X'].min(),
                        df_merged['Y'].max()-df_merged['Y'].min(),
                        df_merged['Z'].max()-df_merged['Z'].min()) / 2.0
        mid_x = (df_merged['X'].max()+df_merged['X'].min()) * 0.5
        mid_y = (df_merged['Y'].max()+df_merged['Y'].min()) * 0.5
        mid_z = (df_merged['Z'].max()+df_merged['Z'].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    except:
        pass

    plt.show()
    print("🎉")

except Exception as e:
    print(f"❌: {e}")

    import traceback
    traceback.print_exc()

In [ ]:
import pandas as pd
import plotly.express as px


coords_file = 'x-beams withcoordinates.txt'
map_file    = 'node_solid_map new.txt'

try:
    print("1. 读取并合并数据...")

    df_coords = pd.read_csv(coords_file, sep='\t')
    df_coords.columns = [c.strip() for c in df_coords.columns]

    df_map = pd.read_csv(map_file, sep=',', skipinitialspace=True)
    df_map.columns = [c.strip() for c in df_map.columns]


    df_coords['Node'] = df_coords['Node'].astype(int)
    df_map['NodeID']  = df_map['NodeID'].astype(int)


    df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')


    if 'BodyID' in df_merged.columns:
        df_merged = df_merged[df_merged['BodyID'] != 0]

        df_merged['BodyID'] = df_merged['BodyID'].astype(str)

    print(f"✅ 准备绘图 (共 {len(df_merged)} 个点)...")


    fig = px.scatter_3d(df_merged,
                        x='X',
                        y='Y',
                        z='Z',
                        color='BodyID',
                        opacity=1.0,
                        title="Interactive X-Beam Visualization (Drag to Rotate)",
                        width=1000,
                        height=800)


    fig.update_traces(marker=dict(size=3))


    fig.update_layout(showlegend=False)


    fig.update_layout(scene=dict(aspectmode='data'))

    print("🎉")
    fig.show()

except Exception as e:
    print(f"❌: {e}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


df_coords = pd.read_csv('x-beams withcoordinates new.txt', sep='\t')
df_coords.columns = [c.strip() for c in df_coords.columns]


df_map = pd.read_csv('node_solid_map new.txt', sep=',', skipinitialspace=True)
df_map.columns = [c.strip() for c in df_map.columns]


df_coords['Node'] = df_coords['Node'].astype(int)
df_map['NodeID'] = df_map['NodeID'].astype(int)


df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')


df_merged = df_merged.sort_values(by='BodyID')


output_df = df_merged[['BodyID', 'Node', 'X', 'Y', 'Z']]


output_df.to_csv('merged_body_nodes.txt', sep='\t', index=False)
print("✅: merged_body_nodes.txt")


def visualize_body(target_body_id):

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')


    ax.scatter(df_coords['X'], df_coords['Y'], df_coords['Z'],
               c='grey', s=1, alpha=0.5, label='Background')


    target_data = df_merged[df_merged['BodyID'] == target_body_id]

    if not target_data.empty:
        ax.scatter(target_data['X'], target_data['Y'], target_data['Z'],
                   c='red', s=20, alpha=1.0, label=f'Body {int(target_body_id)}')


        mid_x = target_data['X'].mean()
        mid_y = target_data['Y'].mean()
        mid_z = target_data['Z'].mean()

        print(f"📍 find Body {target_body_id}，contains {len(target_data)} nodes")
    else:
        print(f"⚠️  BodyID 为 {target_body_id} ")

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Highlight: Body {target_body_id}')
    ax.legend()
    plt.show()


sample_id = df_merged['BodyID'].values[0]


visualize_body(-340693)

#match x-beam bodys' names

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np


data_file = 'xbeamID_nodes.txt'


name_file = 'wb_name_map new.txt'


print("...")

df_nodes = pd.read_csv(data_file, sep='\t')
df_nodes.columns = [c.strip() for c in df_nodes.columns]


df_centers = df_nodes.groupby('BodyID')[['X', 'Y', 'Z']].mean().reset_index()

df_centers.columns = ['BodyID', 'ID_X', 'ID_Y', 'ID_Z']

print(f"   已计算 {len(df_centers)} 个内部 ID 的中心坐标。")


try:
    df_names = pd.read_csv(name_file, sep=',')
    print(f"2. 读取名字映射表... 成功加载 {len(df_names)} 个部件名。")
except:
    print("⚠️ 警告：找不到 wb_name_map.txt。无法显示名字，只能显示 ID。")
    df_names = pd.DataFrame()


id_to_name_map = {}

if not df_names.empty:
    print("3. 正在进行几何匹配 (Matching Geometry)...")
    from scipy.spatial import cKDTree


    id_points = df_centers[['ID_X', 'ID_Y', 'ID_Z']].values
    tree = cKDTree(id_points)


    wb_points = df_names[['Centroid_X', 'Centroid_Y', 'Centroid_Z']].values


    distances, indices = tree.query(wb_points)


    for i, idx in enumerate(indices):
        dist = distances[i]

        if dist < 0.01:
            matched_id = df_centers.iloc[idx]['BodyID']
            part_name  = df_names.iloc[i]['Name']
            id_to_name_map[matched_id] = part_name
        else:
            print(f"   ⚠️ 未匹配到: {df_names.iloc[i]['Name']} (最近距离 {dist:.4f})")

print(f"   匹配完成！共关联了 {len(id_to_name_map)} 个部件。")


def find_and_plot(query):
    """
    query: 可以是 BodyID (数字)，也可以是名字 (字符串)
    """
    target_id = None
    target_name = "Unknown"


    if isinstance(query, str):

        for bid, name in id_to_name_map.items():
            if query in name:
                target_id = bid
                target_name = name
                break
        if target_id is None:
            print(f"❌ 找不到包含 '{query}' 的部件名字。")
            return
    else:

        target_id = query
        target_name = id_to_name_map.get(target_id, "Unnamed Part")


    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')


    ax.scatter(df_nodes['X'], df_nodes['Y'], df_nodes['Z'],
               c='lightgrey', s=1, alpha=0.5)


    target_data = df_nodes[df_nodes['BodyID'] == target_id]

    if not target_data.empty:
        ax.scatter(target_data['X'], target_data['Y'], target_data['Z'],
                   c='red', s=20, alpha=1.0, label=f'{target_name}\n(ID: {int(target_id)})')


        mid_x, mid_y, mid_z = target_data[['X','Y','Z']].mean()
        range_ = 0.03
        ax.set_xlim(mid_x - range_, mid_x + range_)
        ax.set_ylim(mid_y - range_, mid_y + range_)
        ax.set_zlim(mid_z - range_, mid_z + range_)

        print(f"📍 已定位部件: {target_name} (ID: {int(target_id)})")
    else:
        print(f"❌ 数据中找不到 ID {target_id}")

    ax.legend()
    plt.title(f"Visualizing: {target_name}")
    plt.show()


sample_id = df_centers.iloc[0]['BodyID']
find_and_plot("Body_40")

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree


node_file = 'merged_body_nodes.txt'
name_file = 'wb_name_map new.txt'
output_file = 'final_named_structure.txt'

print("1. 读取文件...")
try:

    df_nodes = pd.read_csv(node_file, sep='\t')
    df_nodes.columns = [c.strip() for c in df_nodes.columns]


    df_names = pd.read_csv(name_file, sep=',')
    df_names.columns = [c.strip() for c in df_names.columns]

    print(f"   节点数据: {len(df_nodes)} 行")
    print(f"   名字数据: {len(df_names)} 个部件")

except Exception as e:
    print(f"❌ 读取失败: {e}")
    print("   请确保 merged_body_nodes.txt 和 wb_name_map.txt 都在当前文件夹中！")
    exit()

print("2. 计算几何中心并匹配...")


df_centers = df_nodes.groupby('BodyID')[['X', 'Y', 'Z']].mean().reset_index()
apdl_points = df_centers[['X', 'Y', 'Z']].values


wb_points = df_names[['Centroid_X', 'Centroid_Y', 'Centroid_Z']].values
wb_names = df_names['Name'].values


tree = cKDTree(apdl_points)


dists, idxs = tree.query(wb_points)


id_name_map = {}
matched_count = 0


tolerance = 0.001

for i, idx in enumerate(idxs):
    dist = dists[i]
    if dist < tolerance:

        found_body_id = df_centers.iloc[idx]['BodyID']
        found_name = wb_names[i]
        id_name_map[found_body_id] = found_name
        matched_count += 1
    else:
        print(f"   ⚠️ 未匹配警告: {wb_names[i]} 距离最近的网格中心太远 ({dist:.4f})")

print(f"✅ 匹配完成！成功关联了 {matched_count} / {len(df_names)} 个部件。")


print("3. 生成导出数据...")


df_nodes['BodyName'] = df_nodes['BodyID'].map(id_name_map)


df_final = df_nodes.dropna(subset=['BodyName'])


df_final = df_final.sort_values(by=['BodyName', 'Node'])


output_df = df_final[['BodyName', 'Node', 'X', 'Y', 'Z']]


output_df.to_csv(output_file, sep='\t', index=False)

print(f"🎉 {output_file}")
print("-" * 30)
print("文件预览 (前 10 行):")
print(output_df.head(10))

#FINAL result

In [ ]:
import pandas as pd
import plotly.express as px


filename = "final_named_structure.txt"

try:
    print("1. 读取数据...")
    df = pd.read_csv(filename, sep='\t')

    df.columns = [c.strip() for c in df.columns]

    print(f"   成功读取 {len(df)} 个节点。")
    print(f"   包含部件数量: {len(df['BodyName'].unique())}")

    print("2. 生成交互式图表...")


    fig = px.scatter_3d(
        df,
        x='X',
        y='Y',
        z='Z',
        color='BodyName',
        hover_name='BodyName',
        hover_data={
            'BodyName': False,
            'Node': True,
            'X': True,
            'Y': True,
            'Z': True
        },
        title='Interactive Structure Visualization (Hover to see Name)',
        opacity=1.0,
        width=1200,
        height=900
    )


    fig.update_traces(marker=dict(size=4))


    if len(df['BodyName'].unique()) > 20:
        fig.update_layout(showlegend=False)
        print("   (提示：部件较多，已自动隐藏图例以保持视野清晰)")


    fig.update_layout(scene=dict(aspectmode='data'))


    fig.update_layout(hoverlabel=dict(
        bgcolor="white",
        font_size=14,
        font_family="Rockwell"
    ))

    print("🎉 图表已生成！请在下方查看。")
    fig.show()

except Exception as e:
    print(f"❌ 发生错误: {e}")